# Neural Networks with PyTorch

A practical introduction to building, training, and validating a neural network with PyTorch.

This notebook builds a small neural network from scratch using `torch.nn.Module` and trains it to approximate a nonlinear function.

The goal is not to hide the mechanics behind a high-level trainer. The model, loss, optimizer, batches, training loop, and validation process remain explicit.

> **Understand first. Automate the repetitive work second.**


## Learning Objectives

By the end of this notebook, I should understand:

- how `nn.Module` defines a neural network
- how `nn.Linear` layers transform feature dimensions
- how ReLU introduces nonlinearity
- how trainable parameters are registered
- how `TensorDataset` and `DataLoader` provide mini-batches
- the difference between an epoch, batch, and optimization step
- how a standard PyTorch training loop works
- why `model.train()` and `model.eval()` exist
- why validation must not update model parameters
- how `torch.no_grad()` is used during evaluation
- why a GPU is not automatically faster for a tiny workload


## 1. Imports

We use PyTorch for tensors, neural-network building blocks, optimization, and data loading.

`scikit-learn` is used only for the standard utility of splitting the dataset into training and validation sets.


In [1]:
import time

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split


## 2. Device

For this small experiment, CPU execution is sufficient.

The device is selected explicitly so the same code structure can later be extended to a supported accelerator.

For this notebook, the model and data remain on the CPU because the workload is tiny and GPU overhead would dominate the computation.


In [2]:
device = torch.device("cpu")
print(f"Device: {device}")


Device: cpu


## 3. Define the Neural Network

The model contains:

```text
Input: 1 feature
    ↓
Linear(1 → 16)
    ↓
ReLU
    ↓
Linear(16 → 1)
    ↓
Output: 1 value
```

The hidden layer provides a 16-dimensional intermediate representation.

The ReLU activation is essential here because without a nonlinear activation, stacking linear layers would still produce an overall linear transformation.


In [3]:
class MyNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(1, 16)
        self.layer2 = nn.Linear(16, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = torch.relu(x)
        x = self.layer2(x)
        return x


## 4. Instantiate and Inspect the Model

`nn.Module` automatically registers child modules and their trainable parameters.

This is why `model.parameters()` can later provide the optimizer with all weights and biases.


In [4]:
model = MyNetwork().to(device)

print(model)

print("\nTrainable parameters:")
for name, parameter in model.named_parameters():
    print(f"{name}: {tuple(parameter.shape)}")


MyNetwork(
  (layer1): Linear(in_features=1, out_features=16, bias=True)
  (layer2): Linear(in_features=16, out_features=1, bias=True)
)

Trainable parameters:
layer1.weight: (16, 1)
layer1.bias: (16,)
layer2.weight: (1, 16)
layer2.bias: (1,)


## 5. Create the Dataset

The target relationship is:

$$
y = \sin(x)
$$

This is deliberately nonlinear. It gives the ReLU network something more interesting to learn than the earlier linear example.

We create 200 input/target pairs.


In [5]:
x = torch.linspace(-3.14, 3.14, 200).unsqueeze(1)
target = torch.sin(x)

x = x.to(device)
target = target.to(device)


## 6. Train / Validation Split

The training set is used to update the model parameters.

The validation set is used only to measure how the current model performs on examples that were not used for those parameter updates.

An 80/20 split is used here for this small experiment.


In [6]:
train_x, val_x, train_target, val_target = train_test_split(
    x,
    target,
    test_size=0.2,
    random_state=42
)

print(f"Training examples:   {len(train_x)}")
print(f"Validation examples: {len(val_x)}")


Training examples:   160
Validation examples: 40


## 7. Create a Dataset and DataLoader

`TensorDataset` keeps each input paired with its target.

`DataLoader` handles the repetitive work of creating mini-batches and shuffling the training examples.

With 160 training examples and a batch size of 20:

$$
160 \div 20 = 8
$$

So each epoch contains 8 mini-batches.


In [7]:
dataset = TensorDataset(train_x, train_target)

loader = DataLoader(
    dataset,
    batch_size=20,
    shuffle=True
)

batch_x, batch_target = next(iter(loader))

print(f"Batch inputs:  {batch_x.shape}")
print(f"Batch targets: {batch_target.shape}")


Batch inputs:  torch.Size([20, 1])
Batch targets: torch.Size([20, 1])


## 8. Loss Function and Optimizer

This is a regression problem, so Mean Squared Error is appropriate.

The optimizer receives `model.parameters()`, which contains the registered trainable weights and biases.

The learning cycle will be:

```text
zero_grad
    ↓
forward pass
    ↓
loss
    ↓
backward
    ↓
optimizer.step()
```


In [8]:
loss_fn = nn.MSELoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)


## 9. Training Loop

### Definitions

- **Batch:** a subset of the training examples processed together.
- **Step:** one optimizer update.
- **Epoch:** one complete pass through the training dataset.

The validation set is deliberately excluded from the backward pass.

`model.train()` selects training mode.

The current model contains only `Linear` and ReLU layers, so its forward behavior is the same in both modes. The distinction becomes important for layers such as Dropout and BatchNorm.


In [9]:
epochs = 2000
print_every = 100

start_time = time.time()

for epoch in range(epochs):
    model.train()

    for batch_x, batch_target in loader:
        prediction = model(batch_x)
        train_loss = loss_fn(prediction, batch_target)

        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

    if epoch % print_every == 0:
        print(f"Epoch {epoch:4d} | Training loss: {train_loss.item():.6f}")

training_time = time.time() - start_time
print(f"Training time: {training_time:.2f} seconds")


Epoch    0 | Training loss: 0.447641
Epoch  100 | Training loss: 0.044154
Epoch  200 | Training loss: 0.017052
Epoch  300 | Training loss: 0.014361
Epoch  400 | Training loss: 0.002053
Epoch  500 | Training loss: 0.004050
Epoch  600 | Training loss: 0.002826
Epoch  700 | Training loss: 0.001271
Epoch  800 | Training loss: 0.001097
Epoch  900 | Training loss: 0.000706
Epoch 1000 | Training loss: 0.000505
Epoch 1100 | Training loss: 0.000754
Epoch 1200 | Training loss: 0.000299
Epoch 1300 | Training loss: 0.000340
Epoch 1400 | Training loss: 0.000285
Epoch 1500 | Training loss: 0.000339
Epoch 1600 | Training loss: 0.000732
Epoch 1700 | Training loss: 0.000368
Epoch 1800 | Training loss: 0.000456
Epoch 1900 | Training loss: 0.000380
Training time: 2.55 seconds


## 10. Validation

Validation answers a different question from training:

> How well does the current model perform on data that was not used to update its parameters?

During validation:

- no `backward()` is called
- no optimizer step is performed
- parameters do not change
- `torch.no_grad()` prevents PyTorch from building a gradient graph
- `model.eval()` selects evaluation mode


In [10]:
model.eval()

with torch.no_grad():
    val_prediction = model(val_x)
    val_loss = loss_fn(val_prediction, val_target)

print(f"Validation loss: {val_loss.item():.6f}")


Validation loss: 0.000331


## 11. Inspect the Predictions

A low validation loss is useful, but it is also valuable to inspect what the model actually predicts.



In [11]:
with torch.no_grad():
    predictions = model(x)

for i in range(0, len(x), 40):
    print(
        f"x={x[i].item(): .3f} | "
        f"target={target[i].item(): .3f} | "
        f"prediction={predictions[i].item(): .3f}"
    )


x=-3.140 | target=-0.002 | prediction=-0.042
x=-1.878 | target=-0.953 | prediction=-0.955
x=-0.615 | target=-0.577 | prediction=-0.568
x= 0.647 | target= 0.603 | prediction= 0.609
x= 1.909 | target= 0.943 | prediction= 0.931


## 12. What I Learned

### Neural network

A neural network is a composition of parameterized transformations and nonlinearities:

$$
h = \operatorname{ReLU}(W_1x + b_1)
$$

$$
y = W_2h + b_2
$$

### Training

The model learns by repeatedly:

$$
\text{forward} \rightarrow \text{loss} \rightarrow \text{backward} \rightarrow \text{parameter update}
$$

### Width

Increasing the number of hidden neurons gives a layer more capacity to construct an intermediate representation.

### Depth

Additional nonlinear layers allow successive transformations of learned representations. More depth does not automatically mean better performance; optimization and architecture both matter.

### Validation

Training loss measures performance on data used to update parameters.

Validation loss measures performance on held-out data and helps detect problems such as overfitting.

### PyTorch abstraction

`nn.Module` provides the structure that lets PyTorch register submodules and parameters, expose them through `model.parameters()`, and integrate them with optimizers.

### Final mental model

```text
Dataset
   ↓
DataLoader
   ↓
Mini-batch
   ↓
Model
   ↓
Prediction
   ↓
Loss
   ↓
Autograd
   ↓
Gradients
   ↓
Optimizer
   ↓
Updated parameters
   ↓
Repeat
```

The framework automates the repetitive machinery, but the underlying computation remains understandable.


## Chapter Status

**Neural Networks — Fundamentals: Complete**

The next chapter will move from regression to **classification**, introducing logits, class predictions, and cross-entropy loss.
